# 01 · Data Exploration
**Epidemiological Pulse – Population Health Hotspot Detection**

This notebook walks through the synthetic datasets produced by `data/generate_synthetic_data.py`.
We will:
1. Load all five CSV files
2. Inspect shapes, dtypes, and missing values
3. Visualise seasonal patterns and outbreak spikes
4. Examine cross-signal correlations
5. Produce a data-quality summary


In [ ]:
import sys, warnings
sys.path.insert(0, '..')  # make src/ importable
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

DATA_DIR = Path('../data/synthetic')
print('Data directory:', DATA_DIR.resolve())

## 1 · Load Datasets

In [ ]:
pharmacy   = pd.read_csv(DATA_DIR / 'pharmacy_sales.csv',       parse_dates=['date'])
sentiment  = pd.read_csv(DATA_DIR / 'social_media_sentiment.csv', parse_dates=['date'])
weather    = pd.read_csv(DATA_DIR / 'weather.csv',              parse_dates=['date'])
aqi        = pd.read_csv(DATA_DIR / 'aqi.csv',                  parse_dates=['date'])
er_visits  = pd.read_csv(DATA_DIR / 'er_visits.csv',            parse_dates=['date'])
master     = pd.read_csv(DATA_DIR / 'master_dataset.csv',       parse_dates=['date'])

datasets = {
    'pharmacy':  pharmacy,
    'sentiment': sentiment,
    'weather':   weather,
    'aqi':       aqi,
    'er_visits': er_visits,
}

for name, df in datasets.items():
    print(f'{name:>12s}  shape={df.shape}  date_range=[{df.date.min().date()} .. {df.date.max().date()}]')

## 2 · Basic Statistics & Missing Values

In [ ]:
print('=== MASTER DATASET ===')
display(master.describe(include='all').T)

print('\n=== MISSING VALUES ===')
missing = master.isnull().sum()
display(missing[missing > 0] if missing.any() else pd.Series({'(none)': 0}))

## 3 · Seasonal Patterns

In [ ]:
# Pick a representative ZIP code
zip_code = master['zip_code'].unique()[0]
df_zip = master[master['zip_code'] == zip_code].sort_values('date')

fig = make_subplots(
    rows=4, cols=1,
    shared_xaxes=True,
    subplot_titles=[
        'Pharmacy Sales (OTC units)',
        'Social Media Health Sentiment',
        'Temperature (°F) & AQI',
        'ER Visits'
    ],
    vertical_spacing=0.07
)

fig.add_trace(go.Scatter(x=df_zip['date'], y=df_zip['pharmacy_sales'],
                         name='Pharmacy', line=dict(color='steelblue')), row=1, col=1)
fig.add_trace(go.Scatter(x=df_zip['date'], y=df_zip['sentiment_score'],
                         name='Sentiment', line=dict(color='darkorange')), row=2, col=1)
fig.add_trace(go.Scatter(x=df_zip['date'], y=df_zip['temperature'],
                         name='Temp', line=dict(color='tomato')), row=3, col=1)
fig.add_trace(go.Scatter(x=df_zip['date'], y=df_zip['aqi'],
                         name='AQI', line=dict(color='mediumseagreen')), row=3, col=1)
fig.add_trace(go.Scatter(x=df_zip['date'], y=df_zip['er_visits'],
                         name='ER Visits', line=dict(color='mediumpurple')), row=4, col=1)

fig.update_layout(height=900, title_text=f'All Signals – ZIP {zip_code}',
                  template='plotly_dark', showlegend=True)
fig.show()

## 4 · Outbreak Spike Detection

In [ ]:
# Z-score anomaly on pharmacy sales
df_zip = df_zip.copy()
df_zip['pharmacy_zscore'] = (
    (df_zip['pharmacy_sales'] - df_zip['pharmacy_sales'].rolling(30).mean())
    / df_zip['pharmacy_sales'].rolling(30).std()
).fillna(0)

anomalies = df_zip[df_zip['pharmacy_zscore'].abs() > 2.5]

fig = go.Figure()
fig.add_trace(go.Scatter(x=df_zip['date'], y=df_zip['pharmacy_sales'],
                         name='Pharmacy Sales', line=dict(color='steelblue')))
fig.add_trace(go.Scatter(x=anomalies['date'], y=anomalies['pharmacy_sales'],
                         mode='markers', name='Anomaly',
                         marker=dict(color='red', size=10, symbol='x')))

fig.update_layout(title=f'Pharmacy Sales Anomalies (Z > 2.5) – ZIP {zip_code}',
                  template='plotly_dark')
fig.show()
print(f'Found {len(anomalies)} anomaly dates')

## 5 · Cross-Signal Correlation

In [ ]:
numeric_cols = ['pharmacy_sales', 'sentiment_score', 'temperature', 'aqi', 'er_visits']
corr = df_zip[numeric_cols].corr()

fig = px.imshow(
    corr,
    text_auto='.2f',
    color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1,
    title=f'Signal Correlation Matrix – ZIP {zip_code}',
    template='plotly_dark'
)
fig.show()

## 6 · Cross-ZIP Comparison

In [ ]:
# Average pharmacy sales per ZIP
avg_by_zip = (
    master.groupby('zip_code')['pharmacy_sales']
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)

fig = px.bar(
    avg_by_zip,
    x='zip_code', y='pharmacy_sales',
    title='Average Daily Pharmacy Sales by ZIP Code',
    color='pharmacy_sales',
    color_continuous_scale='Reds',
    template='plotly_dark'
)
fig.show()

## 7 · Data Quality Report

In [ ]:
report_rows = []
for name, df in datasets.items():
    report_rows.append({
        'dataset':     name,
        'rows':        len(df),
        'columns':     len(df.columns),
        'zip_codes':   df['zip_code'].nunique() if 'zip_code' in df.columns else 'N/A',
        'date_range':  f'{df.date.min().date()} → {df.date.max().date()}',
        'missing_pct': f"{df.isnull().mean().mean() * 100:.2f} %",
    })

report = pd.DataFrame(report_rows)
display(report)

print('\n✅ Data exploration complete.')